# Feature engineering

In [1]:
from pyprojroot import here
print("Project root:", here())

Project root: C:\Users\hanis\Flight\flight-delay-prediction


In [2]:
import pandas as pd

from sklearn.metrics import average_precision_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

from features import make_features,encode_features

## Preparing Training and Validation Data

#### We will now use the list of features we got after performing feature engineering.

The selected features are separated from the target variable (`DepDel15`) to create the input matrix (`X`) and target vector (`y`).

The training dataset is used for model learning, while the validation dataset is used to evaluate model performance on unseen data. This separation helps assess the model's ability to generalize beyond the training examples.

In [3]:
# Load train and val data
train_df = pd.read_parquet(here("data/processed/train.parquet"))
val_df = pd.read_parquet(here("data/processed/val.parquet"))

X_train,X_val = make_features(train_df,val_df)

X_train, X_val = encode_features(
    X_train,
    X_val
)

In [4]:
y_train = train_df["DepDel15"]
y_val = val_df["DepDel15"]

# X_train and X_val stay as it is

X_train["airline_delay_rate"].describe()

count    1.791079e+06
mean     1.723064e-01
std      3.647137e-02
min      9.975338e-02
25%      1.430693e-01
50%      1.807268e-01
75%      1.858857e-01
max      2.525430e-01
Name: airline_delay_rate, dtype: float64

## Evaluation of the model

To ensure consistent comparison across multiple machine learning models, a reusable evaluation function is defined.

The function:
1. Trains the model using the training dataset.
2. Generates delay probabilities for the validation dataset.
3. Evaluates performance using Precision-Recall Area Under the Curve (PR-AUC).

PR-AUC is chosen because flight delays represent a minority class in the dataset, making it a more informative metric than simple accuracy. A higher PR-AUC indicates better identification of delayed flights while minimizing false alarms.

In [5]:
def evaluate_model(model, X_train, y_train, X_val, y_val):

    model.fit(X_train, y_train)

    probs = model.predict_proba(X_val)[:, 1]

    pr_auc = average_precision_score(
        y_val,
        probs
    )

    return pr_auc

## Model Selection

To establish a reliable benchmark, multiple machine learning algorithms are evaluated on the same dataset and feature set.

The selected models represent different approaches to classification:

- **Logistic Regression**: A simple linear baseline model.
- **Decision Tree**: A rule-based model capable of capturing non-linear relationships.
- **Random Forest**: An ensemble of decision trees that improves robustness and predictive performance.
- **XGBoost**: A gradient boosting algorithm widely regarded as one of the strongest performers on structured tabular datasets.

Using multiple models allows us to compare their strengths and identify the most suitable approach for flight delay prediction.

In [6]:
models = {
    "Dummy": DummyClassifier(strategy="prior"), # gives 0.18 all the time
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(random_state=42)
}

## Model Training and Performance Comparison

Each model is trained using the training dataset and evaluated on the validation dataset using the previously defined evaluation framework.

To ensure a fair comparison:
- All models use the same feature set.
- All models are trained on the same training data.
- All models are evaluated on the same validation data.
- The same performance metric (PR-AUC) is used throughout.

The resulting scores provide an objective comparison of each model's ability to identify delayed flights.

In [7]:
results = []

for name, model in models.items():

    score = evaluate_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val
    )

    results.append({
        "Model": name,
        "PR-AUC": score
    })

results_df = pd.DataFrame(results)

#### Displaying the results:

In [8]:
# Get results in descending order of PR-AUC
print(
    results_df.sort_values(
        "PR-AUC",
        ascending=False
    )
)

                 Model    PR-AUC
4              XGBoost  0.223733
1  Logistic Regression  0.200529
3        Random Forest  0.196289
2        Decision Tree  0.182901
0                Dummy  0.143774
